In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge, RidgeCV, ElasticNet,ElasticNetCV, LassoCV, lasso_path, enet_path
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline

import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_validate, KFold

from ISLP.models import (ModelSpec as MS, summarize , poly)


# Problems with Regression

## Single Measurement

`statsmodels`:

In [ ]:
data = pd.DataFrame({'x': [1.0], 'y': [2.0]})

design = MS(['x'])
X = design.fit_transform(data)
X

In [ ]:

model = sm.OLS(data['y'], X)
results = model.fit()

results.summary()

`sklearn`:

In [ ]:
model = LinearRegression()
model.fit(data[['x']], data['y'])

model.intercept_, model.coef_


No warning from `sklearn` in this example, but the result is reasonable.

## Nuisance Features

Include meany feautres that have no impact on the response:
$$
Y = 1 + 2\cdot X_1 + 0\cdot X_2 + \ldots +0 \cdot X_p + \epsilon
$$

In [ ]:
n = 100
p = 100 # 99 nuisance features + 1 signal feature

# np.random.seed(318)
np.random.seed(2000)

x1 = np.random.uniform(0, 1, size=n)
y = 1 + 2 * x1 + np.random.normal(loc=0, scale=0.1, size=n)
x_rest = np.random.normal(loc=0, scale=0.01, size=(n, p-1))

X_full = np.column_stack([x1, x_rest])  # shape: (n, p)
x_cols = ["x1"] + [f"x{i}" for i in range(2, p + 1)]

df = pd.DataFrame(X_full, columns=x_cols)
df["y"] = y

# fit model with all features
X = df.drop(columns='y')
# response
Y = df['y']                

In [ ]:
X.head()

Using `statsmodels`:

In [ ]:
X1 = sm.add_constant(X)
model = sm.OLS(Y, X1)
results = model.fit()
results.summary()

`sklearn`:

In [ ]:
model = LinearRegression()
model.fit(X, Y)

model.intercept_, model.coef_[0:4]


No warning, but large covariates on the nuisance variables. Very unstable to n oise.

## Correlated Features

Suppose
$$
Y = 1 + 2\cdot X_1 + 3 \cdot X_2 + \epsilon
$$
where $X_2 =2 - X_1$

In [ ]:
n = 100

np.random.seed(318)

x1 = np.random.uniform(0, 1, size=n)
x2 = 2 - x1
y = 1 + 2 * x1  + 3 * x2 + np.random.normal(loc=0, scale=0.1, size=n)
df = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y})

X = df[['x1', 'x2']]
Y = df['y']

`statsmodels`:

In [ ]:
X1 = sm.add_constant(X)
model = sm.OLS(Y, X1)
results = model.fit()
results.summary()

`sklearn`:

In [ ]:
model = LinearRegression()
model.fit(X, Y)
model.intercept_, model.coef_
